In [1]:
import json
import math
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
import mlflow
import mlflow.pytorch
import numpy as np
import tables
import datetime
from evaluation import WaitKTransformerDatasetAdapter, TokenizedSimulMTEvaluator, MTQualityScorer, NLLBSimulMTAdapter

import torch
import tqdm.notebook as tqdm

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_transformer_distillation")

<Experiment: artifact_location='/mlflow/artifacts/1', creation_time=1779186531684, experiment_id='1', last_update_time=1779186531684, lifecycle_stage='active', name='simulmt_waitk_transformer_distillation', tags={}, trace_location=None, workspace='default'>

In [2]:
def log_configs_to_mlflow(model_cfg, train_cfg):
    mlflow.log_params({
        f"model.{k}": v
        for k, v in asdict(model_cfg).items()
    })

    mlflow.log_params({
        f"train.{k}": v
        for k, v in asdict(train_cfg).items()
    })


def log_gpu_memory_to_mlflow(step: int):
    if not torch.cuda.is_available():
        return

    mlflow.log_metrics(
        {
            "gpu.allocated_gb": torch.cuda.memory_allocated() / 1024**3,
            "gpu.reserved_gb": torch.cuda.memory_reserved() / 1024**3,
            "gpu.max_allocated_gb": torch.cuda.max_memory_allocated() / 1024**3,
        },
        step=step,
    )


def save_and_log_checkpoint(
    *,
    path,
    student,
    optimizer,
    scaler,
    model_cfg,
    train_cfg,
    epoch,
    global_step,
    train_time
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save(
        {
            "model_state_dict": student.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "model_cfg": asdict(model_cfg),
            "train_cfg": asdict(train_cfg),
            "epoch": epoch,
            "global_step": global_step,
            "train_time": train_time
        },
        path,
    )

    # mlflow.log_artifact(str(path), artifact_path="checkpoints")

In [3]:
@dataclass
class SimulTransformerConfig:
    vocab_size: int

    d_model: int = 512
    nhead: int = 8

    num_encoder_layers: int = 6
    num_decoder_layers: int = 6

    dim_feedforward: int = 2048
    dropout: float = 0.1
    max_seq_len: int = 32

    pad_token_id: int = 1
    eos_token_id: int = 2

    tie_embeddings: bool = True

@dataclass
class TrainConfig:
    epochs: int = 5
    short_epochs: bool = False
    batches_per_epoch: int = 10000
    batch_size: int = 1
    gradient_accumulation_steps: int = 8
    num_workers: int = 0

    lr: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0

    wait_k: int | list[int] = 5

    use_kl_loss: bool = False
    use_teacher_ce_loss: bool = True
    use_dataset_ce_loss: bool = True

    kl_weight: float = 0.0
    teacher_ce_weight: float = 1.0
    dataset_ce_weight: float = 0.3

    log_dir: str = "./runs/simulmt_waitk"
    save_every_steps: int = 1000

    use_amp: bool = True

In [4]:
class TranslationDataset(torch.utils.data.Dataset):
    """
    Dataset for tokenized translation data.
    """

    def __init__(self, path: str | Path, lazy: bool = False):
        self.path = str(path)
        self._file = None
        self.lazy = lazy
        self.source_ids = None
        self.target_ids = None
        self.source_mask = None
        self.target_mask = None

        if not lazy:
            with tables.open_file(self.path, mode="r") as file:
                self.source_ids = file.root.source_ids.read()
                self.target_ids = file.root.target_ids.read()
                self.source_mask = file.root.source_mask.read()
                self.target_mask = file.root.target_mask.read()
                self.teacher_top32_ids = file.root.teacher_top32_ids.read()
                self.teacher_top32_logits = file.root.teacher_top32_logits.read()
                #self.teacher_top32_token_ids = file.root.teacher_top32_token_ids.read()
                self.synth_ids = file.root.synth_ids.read()
                self.synth_mask = file.root.synth_mask.read()
                

        with tables.open_file(self.path, mode="r") as file:
            self.length = file.root.source_ids.shape[0]

    def _lazy_open(self):
        if self._file is None:
            self._file = tables.open_file(self.path, mode="r")
        return self._file

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.lazy:
            file = self._lazy_open()

            return {
                "source_ids": torch.as_tensor(file.root.source_ids[idx], dtype=torch.long),
                "target_ids": torch.as_tensor(file.root.target_ids[idx], dtype=torch.long),
                "source_mask": torch.as_tensor(file.root.source_mask[idx], dtype=bool),
                "target_mask": torch.as_tensor(file.root.target_mask[idx], dtype=bool),
                "teacher_top32_ids": torch.as_tensor(file.root.teacher_top32_ids[idx], dtype=torch.long)[..., :-1, :], # i fucked up and i have saved last token predictions too
                "teacher_top32_logits": torch.as_tensor(file.root.teacher_top32_logits[idx], dtype=torch.float32)[..., :-1, :],
                "synth_ids": torch.as_tensor(file.root.synth_ids[idx], dtype=torch.long),
                "synth_mask": torch.as_tensor(file.root.synth_mask[idx], dtype=bool)
            }
        
        return {
            "source_ids": torch.tensor(self.source_ids[idx], dtype=torch.long),
            "target_ids": torch.tensor(self.target_ids[idx], dtype=torch.long),
            "source_mask": torch.tensor(self.source_mask[idx], dtype=bool),
            "target_mask": torch.tensor(self.target_mask[idx], dtype=bool),
            "teacher_top32_ids": torch.as_tensor(self.teacher_top32_ids[idx], dtype=torch.long)[..., :-1, :],
            "teacher_top32_logits": torch.as_tensor(self.teacher_top32_logits[idx], dtype=torch.float32)[..., :-1, :],
            "synth_ids": torch.as_tensor(self.synth_ids[idx], dtype=torch.long),
            "synth_mask": torch.as_tensor(self.synth_mask[idx], dtype=bool)
        }

    def close(self):
        if self._file is not None:
            self._file.close()
            self._file = None

In [5]:
class PositionalEncoding(torch.nn.Module):
    """
    Learnable positional embeddings.
    """

    def __init__(
        self,
        d_model: int,
        max_len: int,
        dropout: float
    ):
        super().__init__()

        self.dropout = torch.nn.Dropout(dropout)

        self.position_embedding = torch.nn.Embedding(
            num_embeddings=max_len,
            embedding_dim=d_model,
        )

        self.max_len = max_len

        self._reset_parameters()

    def _reset_parameters(self):
        """
        Initialize positional embeddings with a small normal distribution.
        """
        torch.nn.init.normal_(
            self.position_embedding.weight,
            mean=0.0,
            std=0.02,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x:
                [batch, seq_len, d_model]

        Returns:
            Tensor with added positional embeddings.
        """

        x = x + self.position_embedding(torch.arange(
            x.shape[1],
            device=x.device,
        ).unsqueeze(0))

        return self.dropout(x)

In [6]:
class WaitKTransformerMT(torch.nn.Module):
    """
    Encoder-decoder Transformer for honest wait-k SimulMT training.

    Training mode:
        - The encoder receives the full source sequence.
        - The encoder uses a causal source-side mask.
        - The decoder uses a causal target-side mask.
        - Cross-attention uses a wait-k memory mask.

    Inference mode:
        - The model receives only the currently visible source prefix.
        - Therefore no wait-k cross-attention mask is needed.
        - The encoder can still use a causal mask to reduce train/inference mismatch.
    """

    def __init__(self, cfg: SimulTransformerConfig):
        super().__init__()

        self.cfg = cfg

        self.src_embedding = torch.nn.Embedding(
            cfg.vocab_size,
            cfg.d_model,
            padding_idx=cfg.pad_token_id,
        )

        self.tgt_embedding = torch.nn.Embedding(
            cfg.vocab_size,
            cfg.d_model,
            padding_idx=cfg.pad_token_id,
        )

        self.src_pos = PositionalEncoding(
            d_model=cfg.d_model,
            max_len=cfg.max_seq_len,
            dropout=cfg.dropout,
        )

        self.tgt_pos = PositionalEncoding(
            d_model=cfg.d_model,
            max_len=cfg.max_seq_len,
            dropout=cfg.dropout,
        )

        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.nhead,
            dim_feedforward=cfg.dim_feedforward,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )

        decoder_layer = torch.nn.TransformerDecoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.nhead,
            dim_feedforward=cfg.dim_feedforward,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )

        self.encoder = torch.nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=cfg.num_encoder_layers,
            norm=torch.nn.LayerNorm(cfg.d_model),
        )

        self.decoder = torch.nn.TransformerDecoder(
            decoder_layer=decoder_layer,
            num_layers=cfg.num_decoder_layers,
            norm=torch.nn.LayerNorm(cfg.d_model),
        )

        self.lm_head = torch.nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.tgt_embedding.weight

        self._init_weights()

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)

            elif isinstance(module, torch.nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=self.cfg.d_model ** -0.5)
                if module.padding_idx is not None:
                    torch.nn.init.zeros_(module.weight[module.padding_idx])

    @staticmethod
    def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
        """
        Boolean causal mask for PyTorch Transformer modules.

        True means "forbidden".
        """
        return torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=device),
            diagonal=1,
        )

    @staticmethod
    def make_waitk_memory_mask(
        tgt_len: int,
        src_len: int,
        k: int,
        device: torch.device
    ) -> torch.Tensor:
        """
        Cross-attention wait-k mask.

        For decoder position j:
            j = 0 predicts the first target token after the decoder prefix.

        Allowed source length:
            visible_src_len = k + j

        True means "forbidden".
        """

        return ~(torch.arange(src_len, device=device)[None, :] < (k + torch.arange(tgt_len, device=device)[:, None]))

    def encode(
        self,
        source_ids: torch.Tensor,
        source_mask: torch.Tensor | None = None,
        *,
        causal: bool = True
    ) -> torch.Tensor:
        """
        Args:
            source_ids:
                [batch, src_len]

            source_mask:
                [batch, src_len], 1 for valid tokens, 0 for padding.

            causal:
                If True, source token i cannot attend to future source tokens.
        """
        if source_mask is None:
            source_key_padding_mask = source_ids.eq(self.cfg.pad_token_id)
        else:
            source_key_padding_mask = ~(source_mask.bool())

        src_attn_mask = None
        if causal:
            src_attn_mask = self.make_causal_mask(source_ids.shape[1], source_ids.device)

        x = self.src_embedding(source_ids) * math.sqrt(self.cfg.d_model)
        x = self.src_pos(x)

        memory = self.encoder(
            src=x,
            mask=src_attn_mask,
            src_key_padding_mask=source_key_padding_mask
        )

        return memory

    def decode(
        self,
        target_input_ids: torch.Tensor,
        memory: torch.Tensor,
        target_input_mask: torch.Tensor | None = None,
        source_mask: torch.Tensor | None = None,
        memory_mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        """
        Args:
            target_input_ids:
                [batch, tgt_len]

            memory:
                [batch, src_len, d_model]

            target_input_mask:
                [batch, tgt_len], 1 for valid tokens, 0 for padding.

            source_mask:
                [batch, src_len], 1 for valid tokens, 0 for padding.

            memory_mask:
                [tgt_len, src_len], True means forbidden.
        """
        if target_input_mask is None:
            target_key_padding_mask = target_input_ids.eq(self.cfg.pad_token_id)
        else:
            target_key_padding_mask = ~(target_input_mask.bool())

        if source_mask is None:
            memory_key_padding_mask = None
        else:
            memory_key_padding_mask = ~(source_mask.bool())

        tgt_mask = self.make_causal_mask(target_input_ids.shape[1], target_input_ids.device)

        y = self.tgt_embedding(target_input_ids) * math.sqrt(self.cfg.d_model)
        y = self.tgt_pos(y)

        hidden = self.decoder(
            tgt=y,
            memory=memory,
            tgt_mask=tgt_mask,
            memory_mask=memory_mask,
            tgt_key_padding_mask=target_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )

        return hidden

    def forward_waitk(
        self,
        source_ids: torch.Tensor,
        target_input_ids: torch.Tensor,
        *,
        k: int,
        source_mask: torch.Tensor | None = None,
        target_input_mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        """
        Honest batched wait-k training forward pass.

        The source is full, but the encoder is causal and cross-attention is
        restricted by the wait-k policy.
        """
        batch_size, src_len = source_ids.shape
        _, tgt_len = target_input_ids.shape

        if source_mask is None:
            source_mask = source_ids.ne(self.cfg.pad_token_id).long()

        if target_input_mask is None:
            target_input_mask = target_input_ids.ne(self.cfg.pad_token_id).long()

        memory = self.encode(
            source_ids=source_ids,
            source_mask=source_mask,
            causal=True,
        )

        memory_mask = self.make_waitk_memory_mask(
            tgt_len=tgt_len,
            src_len=src_len,
            k=k,
            device=source_ids.device,
        )

        hidden = self.decode(
            target_input_ids=target_input_ids,
            memory=memory,
            target_input_mask=target_input_mask,
            source_mask=source_mask,
            memory_mask=memory_mask,
        )

        logits = self.lm_head(hidden)

        return logits

In [7]:
def masked_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    loss = torch.nn.functional.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        reduction="none",
    )

    return (loss.view_as(labels) * mask).sum() / mask.float().sum().clamp_min(1.0)


def masked_kl_divergence(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    mask: torch.Tensor
) -> torch.Tensor:

    student_log_probs = torch.nn.functional.log_softmax(student_logits, dim=-1)
    teacher_probs = torch.nn.functional.softmax(teacher_logits, dim=-1)

    kl = torch.nn.functional.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none",
    ).sum(dim=-1)

    return (kl * mask).sum() / mask.float().sum().clamp_min(1.0)


def masked_topk_kl_divergence(
    *,
    student_logits: torch.Tensor,
    teacher_topk_ids: torch.Tensor,
    teacher_topk_probs: torch.Tensor,
    mask: torch.Tensor
) -> torch.Tensor:
    student_log_probs = torch.nn.functional.log_softmax(
        student_logits,
        dim=-1,
    )

    student_topk_log_probs = student_log_probs.gather(
        dim=-1,
        index=teacher_topk_ids,
    )

    teacher_topk_probs = teacher_topk_probs.float()

    # teacher_topk_probs = teacher_topk_probs / teacher_topk_probs.sum(
    #     dim=-1,
    #     keepdim=True,
    # ).clamp_min(1e-8)

    kl = torch.nn.functional.kl_div(
        student_topk_log_probs,
        teacher_topk_probs,
        reduction="none",
    ).sum(dim=-1)

    return (kl * mask).sum() / mask.float().sum().clamp_min(1.0)


def simulmt_distillation_loss_flexible(
    *,
    student_logits: torch.Tensor,
    dataset_labels: torch.Tensor,
    label_mask: torch.Tensor,

    use_kl_loss: bool,
    use_teacher_ce_loss: bool,
    use_dataset_ce_loss: bool,

    kl_weight: float,
    teacher_ce_weight: float,
    dataset_ce_weight: float,

    teacher_token_ids: torch.Tensor | None = None,
    teacher_student_logits: torch.Tensor | None = None,
    teacher_mask: torch.Tensor | None = None,

    teacher_topk_ids: torch.Tensor | None = None,
    teacher_topk_probs: torch.Tensor | None = None
) -> dict[str, torch.Tensor]:
    device = student_logits.device

    total_loss = torch.zeros([], device=device)

    zero = torch.zeros([], device=device)

    kl_loss = zero
    teacher_ce_loss = zero
    dataset_ce_loss = zero

    if use_kl_loss and kl_weight > 0:
        if teacher_topk_ids is None or teacher_topk_probs is None:
            raise ValueError("Top-k KL is enabled, but teacher_topk_ids/probs are missing.")

        kl_loss = masked_topk_kl_divergence(
            student_logits=student_logits,
            teacher_topk_ids=teacher_topk_ids,
            teacher_topk_probs=teacher_topk_probs,
            mask=label_mask
        )

        total_loss = total_loss + kl_weight * kl_loss

    if use_teacher_ce_loss and teacher_ce_weight > 0:
        if teacher_token_ids is None:
            raise ValueError("Teacher CE is enabled, but teacher_token_ids is missing.")
        teacher_ce_loss = masked_cross_entropy(
            logits=teacher_student_logits,
            labels=teacher_token_ids,
            mask=teacher_mask,
        )

        total_loss = total_loss + teacher_ce_weight * teacher_ce_loss

    if use_dataset_ce_loss and dataset_ce_weight > 0:
        dataset_ce_loss = masked_cross_entropy(
            logits=student_logits,
            labels=dataset_labels,
            mask=label_mask,
        )

        total_loss = total_loss + dataset_ce_weight * dataset_ce_loss

    return {
        "loss": total_loss,
        "kl_loss": kl_loss.detach(),
        "teacher_ce_loss": teacher_ce_loss.detach(),
        "dataset_ce_loss": dataset_ce_loss.detach(),
    }

In [8]:
def prepend_decoder_start_token(
    target_input_ids: torch.Tensor,
    target_input_mask: torch.Tensor,
    *,
    decoder_start_token_id: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Add decoder start token before the existing target prefix.

    Student input:
        [rus_Cyrl, y1, y2, ...]

    Teacher input:
        [</s>, rus_Cyrl, y1, y2, ...]

    The extra teacher output at position 0 must be removed later.
    """
    batch_size = target_input_ids.size(0)
    device = target_input_ids.device

    start_ids = torch.full(
        size=(batch_size, 1),
        fill_value=decoder_start_token_id,
        dtype=target_input_ids.dtype,
        device=device,
    )

    start_mask = torch.ones(
        size=(batch_size, 1),
        dtype=target_input_mask.dtype,
        device=device,
    )

    teacher_target_input_ids = torch.cat(
        [start_ids, target_input_ids],
        dim=1,
    )

    teacher_target_input_mask = torch.cat(
        [start_mask, target_input_mask],
        dim=1,
    )

    return teacher_target_input_ids, teacher_target_input_mask

@torch.no_grad()
def generate_whole_sequence_teacher(
        teacher_model: torch.nn.Module,
        source_ids: torch.Tensor,
        source_mask: torch.Tensor,
        max_len: int,
        *,
        use_amp: bool = True
):
    teacher_model.eval()
    device = source_ids.device
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=use_amp and device.type == "cuda",
    ):
        outputs = teacher_model.generate(
            input_ids=source_ids,
            attention_mask=source_mask,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("rus_Cyrl"),
            max_length=max_len
        )

        if outputs.size(1) < max_len:
            outputs = torch.concat([outputs, torch.full(size=(outputs.size(0), max_len - outputs.size(1)), fill_value=tokenizer.pad_token_id, dtype=outputs.dtype, device=device)], dim=1)

    return outputs

@torch.no_grad()
def generate_next_tokens_teacher(
    teacher_model: torch.nn.Module,
    source_ids: torch.Tensor,
    source_mask: torch.Tensor,
    target_input_ids: torch.Tensor,
    target_mask: torch.Tensor,
    *,
    use_amp: bool = True
) -> dict[str, torch.Tensor]:
    teacher_model.eval()
    device = source_ids.device

    target_input_ids, target_mask = prepend_decoder_start_token(
        target_input_ids,
        target_mask,
        decoder_start_token_id=teacher_model.config.decoder_start_token_id,
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=use_amp and device.type == "cuda",
    ):
        outputs = teacher_model(
            input_ids=source_ids,
            attention_mask=source_mask,
            decoder_input_ids=target_input_ids,
            decoder_attention_mask=target_mask,
            use_cache=False
        ).logits[:, 1:, 0]

    return {
        "logits": outputs,
        "token_ids": outputs.argmax(dim=-1)
    }

@torch.no_grad()
def generate_next_tokens_teacher_topk(
    teacher_model: torch.nn.Module,
    source_ids: torch.Tensor,
    source_mask: torch.Tensor,
    target_input_ids: torch.Tensor,
    target_mask: torch.Tensor,
    *,
    top_k: int,
    use_amp: bool = True
) -> dict[str, torch.Tensor]:
    teacher_model.eval()
    device = source_ids.device

    target_input_ids, target_mask = prepend_decoder_start_token(
        target_input_ids,
        target_mask,
        decoder_start_token_id=teacher_model.config.decoder_start_token_id,
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=use_amp and device.type == "cuda",
    ):
        outputs = teacher_model(
            input_ids=source_ids,
            attention_mask=source_mask,
            decoder_input_ids=target_input_ids,
            decoder_attention_mask=target_mask,
            use_cache=False
        ).logits[:, 1:, :]

    topk_logits, topk_ids = torch.topk(
        outputs,
        k=top_k,
        dim=-1,
    )

    return {
        "topk_logits": topk_logits,
        "topk_ids": topk_ids,
        "token_ids": topk_ids[:, :, 0]
    }

def pad_token_sequence(tensor: torch.Tensor, token: int, pad_len: int) -> torch.Tensor:
    return torch.concat([tensor, torch.full(size=(tensor.size(0), pad_len - tensor.size(1)), fill_value=token, dtype=tensor.dtype, device=tensor.device)], dim=-1)

In [9]:
def train_waitk_student(
    *,
    student: WaitKTransformerMT,
    train_dataset: TranslationDataset,
    model_cfg: SimulTransformerConfig,
    train_cfg: TrainConfig,
    teacher_model: torch.nn.Module,
    device: str | torch.device = "cuda",
    mlflow_run_name: str | None = None
):
    mlflow_run_name = mlflow_run_name or "waitk-student"
    start = datetime.datetime.now()
    with mlflow.start_run(run_name=mlflow_run_name):
        log_configs_to_mlflow(model_cfg, train_cfg)

        mlflow.set_tags({
            "task": "SimulMT",
            "policy": "wait-k",
            "teacher": "NLLB",
            "architecture": "VanillaTransformer"
        })

        device = torch.device(device)
        rng = np.random.default_rng()

        student.to(device)
        student = torch.compile(student, mode="max-autotune")
        teacher_model.to(device)
        teacher_model.eval()

        for p in teacher_model.parameters():
            p.requires_grad_(False)

        optimizer = torch.optim.AdamW(
            student.parameters(),
            lr=train_cfg.lr,
            weight_decay=train_cfg.weight_decay,
            betas=(0.9, 0.98),
        )

        scaler = torch.amp.GradScaler(
            enabled=train_cfg.use_amp and device.type == "cuda"
        )

        global_step = 0
        optimizer_step = 0
        current_dataset = train_dataset
        sampler = None

        for epoch in range(train_cfg.epochs):
            student.train()

            if train_cfg.short_epochs and train_cfg.batches_per_epoch > 0:
                sampler = torch.utils.data.RandomSampler(train_dataset, replacement=False, num_samples=train_cfg.batches_per_epoch * train_cfg.batch_size)

            dataloader = torch.utils.data.DataLoader(
                current_dataset,
                batch_size=train_cfg.batch_size,
                shuffle=not train_cfg.short_epochs,
                num_workers=train_cfg.num_workers,
                sampler=sampler,
                pin_memory=True
            )

            progress = tqdm.tqdm(
                dataloader,
                desc=f"epoch {epoch + 1}/{train_cfg.epochs}",
                leave=True,
            )

            optimizer.zero_grad(set_to_none=True)

            for micro_step, batch in enumerate(progress):
                source_ids = batch["source_ids"].to(device, non_blocking=True)
                target_ids = batch["target_ids"].to(device, non_blocking=True)
                source_mask = batch["source_mask"].to(device, non_blocking=True)
                target_mask = batch["target_mask"].to(device, non_blocking=True)

                target_input_ids = target_ids[:, :-1]
                dataset_labels = target_ids[:, 1:]

                target_input_mask = target_mask[:, :-1]
                label_mask = target_mask[:, 1:]

                teacher_token_ids = None
                teacher_topk_ids = None
                teacher_topk_probs = None
                teacher_mask = None
                teacher_student_logits = None

                with torch.no_grad():
                    if train_cfg.use_kl_loss:
                            teacher_topk_ids = batch["teacher_top32_ids"].to(device, non_blocking=True)
                            teacher_topk_probs = torch.nn.functional.softmax(batch["teacher_top32_logits"].to(device, non_blocking=True), dim=-1)
                    #if train_cfg.use_teacher_ce_loss:
                        #teacher_token_ids = torch.argmax(batch["teacher_top32_logits"].to(device, non_blocking=True), dim=-1)
                        #teacher_token_ids = batch["synth_ids"].to(device, non_blocking=True)[..., 2:]
                        #teacher_token_ids = pad_token_sequence(teacher_token_ids, tokenizer.pad_token_id, 63)
                        #teacher_mask = batch["synth_mask"].to(device, non_blocking=True)[..., 2:]
                        #teacher_mask = pad_token_sequence(teacher_mask, 0, 63)

                if isinstance(train_cfg.wait_k, list):
                    wait_k = rng.choice(train_cfg.wait_k)
                else:
                    wait_k = train_cfg.wait_k
                with torch.amp.autocast(
                    enabled=train_cfg.use_amp and device.type == "cuda",
                    device_type="cuda",
                ):
                    student_logits = student.forward_waitk(
                        source_ids=source_ids,
                        target_input_ids=target_input_ids,
                        k=wait_k,
                        source_mask=source_mask,
                        target_input_mask=target_input_mask,
                    )

                    if train_cfg.use_teacher_ce_loss:
                        teacher_token_ids = pad_token_sequence(batch["synth_ids"].to(device, non_blocking=True)[..., 2:], tokenizer.pad_token_ids, 63)
                        teacher_mask = pad_token_sequence(batch["synth_mask"].to(device, non_blocking=True)[..., 2:], 0, 63)
                        teacher_student_logits = student.forward_waitk(
                            source_ids=source_ids,
                            target_input_ids=teacher_token_ids,
                            k=wait_k,
                            source_mask=source_mask,
                            target_input_mask=teacher_mask
                        )

                    loss_dict = simulmt_distillation_loss_flexible(
                        student_logits=student_logits,
                        dataset_labels=dataset_labels,
                        label_mask=label_mask,

                        use_kl_loss=train_cfg.use_kl_loss,
                        use_teacher_ce_loss=train_cfg.use_teacher_ce_loss,
                        use_dataset_ce_loss=train_cfg.use_dataset_ce_loss,

                        kl_weight=train_cfg.kl_weight,
                        teacher_ce_weight=train_cfg.teacher_ce_weight,
                        dataset_ce_weight=train_cfg.dataset_ce_weight,

                        teacher_token_ids=teacher_token_ids,
                        teacher_mask=teacher_mask,
                        teacher_student_logits=teacher_student_logits,

                        teacher_topk_ids=teacher_topk_ids,
                        teacher_topk_probs=teacher_topk_probs
                    )

                    loss = loss_dict["loss"]
                    loss_for_backward = loss / train_cfg.gradient_accumulation_steps

                scaler.scale(loss_for_backward).backward()

                global_step += 1

                should_step = (
                    global_step % train_cfg.gradient_accumulation_steps == 0
                )

                if should_step:
                    if train_cfg.grad_clip is not None and train_cfg.grad_clip > 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(
                            student.parameters(),
                            train_cfg.grad_clip,
                        )

                    scaler.step(optimizer)
                    scaler.update()

                    optimizer.zero_grad(set_to_none=True)
                    optimizer_step += 1

                metrics = {
                    "epoch": epoch,
                    "optimizer_step": optimizer_step,
                    "loss.total": float(loss.detach().cpu()),
                    "loss.kl": float(loss_dict["kl_loss"].cpu()),
                    "loss.teacher_ce": float(loss_dict["teacher_ce_loss"].cpu()),
                    "loss.dataset_ce": float(loss_dict["dataset_ce_loss"].cpu()),
                    "lr": optimizer.param_groups[0]["lr"]
                }

                mlflow.log_metrics(metrics, step=global_step)

                if global_step % 100 == 0:
                    log_gpu_memory_to_mlflow(global_step)

                progress.set_postfix(
                    loss=f"{metrics['loss.total']:.4f}",
                    kl=f"{metrics['loss.kl']:.4f}",



                    t_ce=f"{metrics['loss.teacher_ce']:.4f}",
                    d_ce=f"{metrics['loss.dataset_ce']:.4f}",
                )
            save_and_log_checkpoint(
                path=f"checkpoints/epoch_{epoch + 1}.pt",
                student=student,
                optimizer=optimizer,
                scaler=scaler,
                model_cfg=model_cfg,
                train_cfg=train_cfg,
                epoch=epoch,
                global_step=global_step,
                train_time=datetime.datetime.now() - start
            )

        # save_and_log_checkpoint(
        #     path=f"checkpoints/final.pt",
        #     student=student,
        #     optimizer=optimizer,
        #     scaler=scaler,
        #     model_cfg=model_cfg,
        #     train_cfg=train_cfg,
        #     epoch=epoch,
        #     global_step=global_step,
        #     train_time=datetime.datetime.now() - start
        # )

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

teacher = AutoModelForSeq2SeqLM.from_pretrained(
    teacher_name,
    dtype=torch.float16,
)

# dataset = TranslationDataset("./data/train_dataset.hdf5")

Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

In [12]:
model_cfg = SimulTransformerConfig(
    vocab_size=len(tokenizer),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    max_seq_len=64,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

train_cfg = TrainConfig(
    epochs=5,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=10,

    use_kl_loss=True,
    use_teacher_ce_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    teacher_ce_weight=1.0,
    dataset_ce_weight=1.0,

    lr=1e-4,
    use_amp=True,
)

student = WaitKTransformerMT(model_cfg)

train_waitk_student(
    student=student,
    train_dataset=dataset,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    teacher_model=teacher,
    device="cuda",
)

epoch 1/5:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 2/5:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 3/5:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 4/5:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 5/5:   0%|          | 0/14504 [00:00<?, ?it/s]

🏃 View run waitk-student at: http://localhost:5000/#/experiments/1/runs/da30c73b168e4bf296c9597de3623de8
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [13]:
def load_training_checkpoint(
    checkpoint_path: str,
    *,
    device: str = "cuda"
):
    """
    Load model, configs, optimizer, scaler and training progress
    from a full training checkpoint.

    Returns:
        student
        optimizer
        scaler
        model_cfg
        train_cfg
        state
        train_time
    """
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model_cfg = SimulTransformerConfig(**checkpoint["model_cfg"])
    train_cfg = TrainConfig(**checkpoint["train_cfg"])
    train_time = checkpoint["train_time"]

    student = torch.compile(WaitKTransformerMT(model_cfg))
    student.load_state_dict(checkpoint["model_state_dict"])
    student.to(device)
    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
        betas=(0.9, 0.98),
    )

    if "optimizer_state_dict" in checkpoint and checkpoint["optimizer_state_dict"] is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    scaler = torch.amp.GradScaler(
        enabled=train_cfg.use_amp and device == "cuda"
    )

    if "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    state = {
        "epoch": checkpoint.get("epoch", 0),
        "global_step": checkpoint.get("global_step", checkpoint.get("step", 0)),
        "optimizer_step": checkpoint.get("optimizer_step", checkpoint.get("step", 0)),
    }

    return student, optimizer, scaler, model_cfg, train_cfg, state, train_time

In [14]:
student, optimizer, scaler, model_cfg, train_cfg, state, train_time = load_training_checkpoint("./checkpoints/epoch_5.pt")

In [12]:
@torch.no_grad()
def translate_with_latency(
    model,
    tokenizer,
    source: str,
    max_len: int = 100,
    k: int = 5,
    speed: int = 1,
    source_lang: str = "eng_Latn",
    target_lang: str = "rus_Cyrl",
) -> str:
    model.eval()
    device = next(model.parameters()).device

    tokenizer.src_lang = source_lang

    inputs = tokenizer(
        source,
        return_tensors="pt",
        truncation=True,
        max_length=model.cfg.max_seq_len,
    ).to(device)

    source_len = int(inputs["attention_mask"].sum().item())
    visible_prefix_len = min(k, source_len, model.cfg.max_seq_len)

    decoder_start_token_id = model.cfg.eos_token_id
    target_lang_id = tokenizer.convert_tokens_to_ids(target_lang)

    target_tokens = torch.tensor(
        [[target_lang_id]],
        dtype=torch.long,
        device=device,
    )

    i = 1

    while target_tokens.size(1) < min(max_len, model.cfg.max_seq_len):
        latency_inputs = inputs["input_ids"][:, :visible_prefix_len]
        latency_attention_mask = inputs["attention_mask"][:, :visible_prefix_len]

        memory = model.encode(
            source_ids=latency_inputs,
            source_mask=latency_attention_mask,
            causal=True,
        )

        target_mask = target_tokens.ne(model.cfg.pad_token_id).long()

        hidden = model.decode(
            memory=memory,
            target_input_ids=target_tokens,
            target_input_mask=target_mask,
            source_mask=latency_attention_mask,
            memory_mask=None,
        )

        logits = model.lm_head(hidden)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)

        if next_token.item() != tokenizer.eos_token_id:
            target_tokens = torch.cat([target_tokens, next_token], dim=-1)

        print(f"Iteration {i}")
        print(f"\tInput: {tokenizer.batch_decode(latency_inputs, skip_special_tokens=False)[0]}")
        print(f"\tTarget: {tokenizer.batch_decode(target_tokens, skip_special_tokens=False)[0]}")
        print(f"\tGenerated token: {tokenizer.batch_decode(next_token, skip_special_tokens=False)[0]}")
        i += 1

        visible_prefix_len = min(
            visible_prefix_len + speed,
            source_len,
            model.cfg.max_seq_len,
        )

        if next_token.item() == tokenizer.eos_token_id and visible_prefix_len >= source_len:
            break

    return tokenizer.batch_decode(
        target_tokens,
        skip_special_tokens=True,
    )[0]

In [25]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Frozen parameters:    {total - trainable:,}")

    return {
        "total": total,
        "trainable": trainable,
        "frozen": total - trainable,
    }

In [26]:
count_parameters(student)

Total parameters:     306,558,976
Trainable parameters: 306,558,976
Frozen parameters:    0


{'total': 306558976, 'trainable': 306558976, 'frozen': 0}

In [41]:
train_time.seconds / 3600

6.004444444444444

In [31]:
#text = "When the old man opened the small wooden door at the end of the garden, he saw a narrow path leading through the wet grass toward a quiet river, where a little girl in a red coat was feeding ducks and laughing because the wind kept turning her yellow umbrella inside out."
#text = "For he did not know that beyond the lake he called home lies a deeper, darker ocean green, where waves are both wilder and more serene, to its ports I've been, to its ports I've been."
#text = "By the time the engineer realized that the lead pipe she had been asked to wind around the old machine was actually meant to record the pressure changes rather than carry water, the supervisor, who had been present during the confusing meeting, refused to permit anyone to object until the final report was read aloud."
text = "EX-View is a proprietary SONY technology in which the P/N junction of each photodiode in the CCD matrix is specially fabricated to have much better photon-to-electron conversion efficiency."
# inputs = tokenizer(text, return_tensors="pt").to("cuda")

In [32]:
translate_with_latency(student, tokenizer, text, 64, 3, 1)

Iteration 1
	Input: eng_Latn EX-
	Target: rus_Cyrl EX
	Generated token: EX
Iteration 2
	Input: eng_Latn EX-V
	Target: rus_Cyrl EX-
	Generated token: -
Iteration 3
	Input: eng_Latn EX-View
	Target: rus_Cyrl EX-V
	Generated token: V
Iteration 4
	Input: eng_Latn EX-View is
	Target: rus_Cyrl EX-View
	Generated token: iew
Iteration 5
	Input: eng_Latn EX-View is a
	Target: rus_Cyrl EX-View -
	Generated token: -
Iteration 6
	Input: eng_Latn EX-View is a propriet
	Target: rus_Cyrl EX-View - это
	Generated token: это
Iteration 7
	Input: eng_Latn EX-View is a proprietary
	Target: rus_Cyrl EX-View - это нес
	Generated token: нес
Iteration 8
	Input: eng_Latn EX-View is a proprietary S
	Target: rus_Cyrl EX-View - это несво
	Generated token: во
Iteration 9
	Input: eng_Latn EX-View is a proprietary SON
	Target: rus_Cyrl EX-View - это несвобо
	Generated token: бо
Iteration 10
	Input: eng_Latn EX-View is a proprietary SONY
	Target: rus_Cyrl EX-View - это несвобод
	Generated token: д
Iteration 11
	Input

'EX-View - это несвободная технология SONY, в которой P/N-ссылка каждого фотодиод в матрице CCD специально изготовлена для того чтобы иметь гораздо лучшее эффективность конверсии фотона.'

In [ ]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=True,
)

adapter = NLLBSimulMTAdapter(
    model=teacher,
    tokenizer=tokenizer,
    name="NLLB_wait_k",
    device="cuda",
    use_amp=True,
    amp_dtype=torch.bfloat16,
    max_source_len=64,
)

evaluator = TokenizedSimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

result = evaluator.evaluate(
    model=adapter,
    dataset=eval_dataset,
    wait_k=5,
    batch_size=512,
    max_new_tokens=63
)

print(result.metrics)

with open("./metrics/nllb_k5.json", "w") as file:
    json.dump(result.metrics, file)

Validating NLLB_wait_k, wait_k=5:   0%|          | 0/605 [00:00<?, ?it/s]

In [ ]:
adapter = WaitKTransformerDatasetAdapter(
    model=teacher,
    tokenizer=tokenizer,
    name="Transformer_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = TokenizedSimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

for k in [3, 7, 9, 11, 13]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63
    )
    
    print(result.metrics)
    
    with open(f"./metrics/transformer_k{k}.json", "w") as file:
        json.dump(result.metrics, file)